# Experiment 4: Real-World Validation on UGR’16 ISP Traffic

Objective:
Evaluate the robustness of DriftGuard under real-world ISP traffic
characterized by label noise, uncontrolled distributions, and
non-stationary behavior.

This experiment is conducted WITHOUT retraining or parameter tuning.


### Experiment Contract

- Models are trained ONLY on CICIDS2017
- No retraining, no fine-tuning on UGR’16
- UGR’16 labels are treated as weak ground truth
- Goal is behavioral analysis, not leaderboard scores
- Degradation is expected and analyzed


In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib
from tensorflow.keras.models import load_model


In [14]:
BASE = "../../data/UGR16/UGR16v2noIRC"

X_train = pd.read_csv(f"{BASE}.Xtrain.csv")
X_test  = pd.read_csv(f"{BASE}.Xtest.csv")

y_train = pd.read_csv(f"{BASE}.Ytrain.csv")
y_test  = pd.read_csv(f"{BASE}.Ytest.csv")

print(X_train.shape, X_test.shape)
print(y_train.value_counts())


(98262, 135) (43200, 135)
Row           labeldos  labelscan11  labelscan44  labelnerisbotnet  labelblacklist  labelanomalyidpscan  labelanomalysshscan  labelanomalyspam
201603190000  0         0            0            0                 163             0                    0                    0                   1
201603190001  0         0            0            0                 217             0                    0                    0                   1
201603190002  0         0            0            0                 808             0                    0                    0                   1
201603190003  0         0            0            0                 327             0                    0                    0                   1
201603190004  0         0            0            0                 137             0                    0                    0                   1
                                                                                           

📌 Important

You will NOT train on X_train

We load it only to understand distributions

Minimal Cleaning (NO FEATURE ENGINEERING)

UGR’16 is already preprocessed. Don’t over-handle it.

In [15]:
X_test = X_test.replace([np.inf, -np.inf], np.nan).dropna()
y_test = y_test.loc[X_test.index]

print("Clean test shape:", X_test.shape)


Clean test shape: (43200, 135)


Feature Alignment (CRITICAL STEP)

UGR’16 ≠ CICIDS feature space.
We must align to what DriftGuard expects.

Load CICIDS feature reference

In [16]:
X_cic_ref = np.load("../../data/X_phase2.npy")
num_features_expected = X_cic_ref.shape[1]

print("Expected feature count:", num_features_expected)
print("UGR feature count:", X_test.shape[1])


Expected feature count: 78
UGR feature count: 135


Strategy (Defensible)

Use numeric features only

If extra columns exist → truncate

If fewer exist → stop (that’s a paper result)

In [17]:
X_test = X_test.select_dtypes(include=[np.number])

if X_test.shape[1] > num_features_expected:
    X_test = X_test.iloc[:, :num_features_expected]
elif X_test.shape[1] < num_features_expected:
    raise ValueError("UGR16 feature space smaller than CICIDS — cannot align safely")


Scaling (Inference-Time Scaling)

⚠️ Yes, we re-fit scaler.
Different environment → different scale.

In [18]:
scaler = StandardScaler()
X_ugr = scaler.fit_transform(X_test)

y_ugr = y_test.values.ravel()

print(X_ugr.shape, y_ugr.shape)


(43200, 78) (388800,)


In [19]:
iforest = joblib.load("../../models/isolation_forest.pkl")
autoencoder = load_model("../../models/autoencoder.keras")
lstm_autoencoder = load_model("../../models/lstm_autoencoder.keras")


Isolation Forest Inference

In [20]:
ugr_iforest_scores = iforest.decision_function(X_ugr)


Autoencoder Inference

In [21]:
X_recon = autoencoder.predict(X_ugr, verbose=0)
ugr_ae_errors = np.mean((X_ugr - X_recon) ** 2, axis=1)


LSTM (ONLY IF TEMPORAL ORDER EXISTS)

UGR’16 is sequential, so this is valid.

In [22]:
def create_sequences(X, window=10):
    return np.array([X[i:i+window] for i in range(len(X) - window)])

WINDOW = 10
X_seq = create_sequences(X_ugr, WINDOW)

X_seq_pred = lstm_autoencoder.predict(X_seq, verbose=0)
ugr_lstm_errors = np.mean((X_seq - X_seq_pred) ** 2, axis=(1,2))


In [23]:
offset = len(X_ugr) - len(ugr_lstm_errors)

ugr_iforest_scores = ugr_iforest_scores[offset:]
ugr_ae_errors = ugr_ae_errors[offset:]
y_ugr = y_ugr[offset:]


Risk Scoring (UNCHANGED FORMULA)

This is crucial — no tuning.

In [24]:
from sklearn.preprocessing import MinMaxScaler

scores = np.vstack([
    -ugr_iforest_scores,
    ugr_ae_errors,
    ugr_lstm_errors
]).T

scores_scaled = MinMaxScaler().fit_transform(scores)

ugr_risk = (
    0.4 * scores_scaled[:, 0] +
    0.3 * scores_scaled[:, 1] +
    0.3 * scores_scaled[:, 2]
)


Behavioral Evaluation (NOT JUST METRICS)

Risk Score Distribution

In [26]:
plt.hist(ugr_risk[y_ugr == 0], bins=50, alpha=0.6, label="Benign")
plt.hist(ugr_risk[y_ugr == 1], bins=50, alpha=0.6, label="Attack")
plt.legend()
plt.title("UGR’16 Risk Score Distribution")
plt.show()


IndexError: boolean index did not match indexed array along axis 0; size of axis is 43190 but size of corresponding boolean axis is 388790